In [151]:
# imports
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler
import os

import sys
from pathlib import Path
BACKTESTER_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(BACKTESTER_ROOT))

from models.gaussian import GaussianMarketRegimeDetector

In [152]:
# notebook settings

pio.templates.default = "plotly_dark"


#### Regime Detection via a Hidden Markov Model

##### Markets do not behave the same at all times, sometimes they **trend calmly**, while at other times they are **aggresively volatile**. 

##### Through the use of a Gaussian HMM I built into my event-driven backtester, I detect what 'regime' the market is in and use the probabilities of this to **scale position sizes**. 

___________________________________________________________________________________________________________________________________________________________________________________________

#### 1.) Feature Engineering

##### I feed the model **5** inputs to allow it to see more than just price movement. 

##### **Rolling features** are created over a 21-day window, using only bars seen and stored in the model at any given time, as intended within my event-driven framework. 

##### They are as follows: mean returns, volatility, skew, 5-day momentum, and log VIX (capturing implied vol alongside realised).


In [153]:
# get_features() function
def get_features(self, fit_scaler):
    bars = self.spy_bar_history
    vix_bars = self.vix_bar_history

    # Align dates of bars.
    bar_dates = {b['date'].date(): b for b in bars}
    vix_dates = {b['date'].date(): b for b in vix_bars}

    same_dates = sorted(set(bar_dates) & set(vix_dates))

    bars = [bar_dates[date] for date in same_dates]
    vix_bars = [vix_dates[date] for date in same_dates]

    assert bars[-1]['date'].date() == vix_bars[-1]['date'].date()




    closes_array = np.array([bar['close'] for bar in bars])
    closes_series = pd.Series(closes_array)

    vix_closes_array = np.array([bar['close'] for bar in vix_bars])
    vix_closes_series = pd.Series(vix_closes_array)

    returns = closes_series.pct_change()
    mean_returns = returns.rolling(window=21).mean()
    vol = returns.rolling(window=21).std()
    skew = returns.rolling(window=21).skew()
    momentum_5_day = closes_series.pct_change(5)
    vix_log = np.log(vix_closes_series)


    df = pd.DataFrame({"Mean_Returns": mean_returns, 
                        "Vol": vol,
                        "Skew": skew, 
                        "5_Day_Momentum": momentum_5_day,
                        "VIX_Log": vix_log})

    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)

    momentum_mean = df['5_Day_Momentum'].mean()
    momentum_std = df['5_Day_Momentum'].std()

    df['5_Day_Momentum'] = np.where(df['5_Day_Momentum'] > momentum_mean + 3 * momentum_std,
                                    momentum_mean + 2 * momentum_std,
                                    df['5_Day_Momentum'])
    df['5_Day_Momentum'] = np.where(df['5_Day_Momentum'] < momentum_mean - 3 * momentum_std,
                                    momentum_mean - 2 * momentum_std,
                                    df['5_Day_Momentum'])
    
    X = df[['Mean_Returns', 'Vol', 'Skew', '5_Day_Momentum', 'VIX_Log']]
    if fit_scaler == True:
        X_new = self.scaler.fit_transform(X)
    else:
        X_new = self.scaler.transform(X)

    return X_new

In [154]:
# populate spy_bars and vix_bars history and simulate feature engineering

spy_csv_dir = "~/python-projects/ed-backtest/backtester/data/sp_constituents/SPY.csv"
vix_csv_dir = "~/python-projects/ed-backtest/backtester/data/sp_constituents/^VIX.csv"

s_path = os.path.join(spy_csv_dir)
v_path = os.path.join(vix_csv_dir)

spy_df = pd.read_csv(s_path, parse_dates=['Date'])
vix_df = pd.read_csv(v_path, parse_dates=['Date'])

spy_bar_history = [
    {'date': pd.Timestamp(row.Date).to_pydatetime(),
     'close': row.Close} for row in spy_df.itertuples()
]

vix_bar_history = [
    {'date': pd.Timestamp(row.Date).to_pydatetime(),
     'close': row.Close} for row in vix_df.itertuples()
]


# Run the get_features function

class DummyRegimeDetector:
    def __init__(self, spy_history, vix_history):
        self.spy_bar_history = spy_history
        self.vix_bar_history = vix_history
        self.scaler = StandardScaler()

    get_features = GaussianMarketRegimeDetector.get_features


detector = DummyRegimeDetector(spy_bar_history, vix_bar_history)
X = detector.get_features(fit_scaler=True)

X = pd.DataFrame(X, columns=['Mean_Returns', 'Vol', 'Skew', 'Momentum_5_Day', 'log_VIX'])
print(X.head())


   Mean_Returns       Vol      Skew  Momentum_5_Day   log_VIX
0     -0.403443 -0.511428 -0.335623        0.702539 -1.181075
1     -0.192672 -0.595238 -0.182214        0.492968 -1.150201
2      0.192418 -0.564704 -0.247698        1.011278 -1.290673
3      0.051875 -0.573538 -0.036600        0.643278 -1.164408
4      0.108825 -0.576083 -0.139945        0.392523 -1.195441


#### 2.) The Model (GaussianHMM)

##### I ran **Bayesian Inference Criterion (BIC)** analysis, which penalises models for having a larger number of features. I settled on 4 components (hidden states), as this produced the lowest BIC and also made the most economic sense.

##### **Full covariance matrix** is used due to the features being correlated, allowing the clusters to capture this.

*While I use a single Gaussian per cluster here, academic papers I've read point to better results from Gaussian Mixture Models. I plan to test this in a future iteration.




In [155]:
# Gaussian Model

model = GaussianHMM(n_components=4, covariance_type="full", n_iter=100, random_state=0)

#### 3.) Which Regime Are We In?

##### The HMM has no knowledge of what 'regime' it is in. It just finds **statistical clusters**.

#### My solution to this is to rank the hidden states by their learned mean returns, volatility, and VIX to map them automatically.

 **BULL** has the highest mean returns coupled with low volatility.

 **BEAR** has the lowest mean returns and high volatility.

 **TRANSITION** has low realised vol and high VIX (implied vol).

 **RECOVERY** has high mean returns and high realised volatility.

 



In [156]:
# Train the model and view raw hidden states


class DummyRegimeDetector:
    def __init__(self, spy_history, vix_history):
        self.spy_bar_history = spy_history
        self.vix_bar_history = vix_history
        self.scaler = StandardScaler()
        self.model = GaussianHMM(n_components=4, covariance_type='full', n_iter=100, random_state=1)
        self.current_probs = None
        self.current_regime = None

    get_features = GaussianMarketRegimeDetector.get_features
    assign_labels = GaussianMarketRegimeDetector.assign_labels
    get_highest_prob_state = GaussianMarketRegimeDetector.get_highest_prob_state




detector = DummyRegimeDetector(spy_bar_history, vix_bar_history)
X = detector.get_features(fit_scaler=True)
detector.model.fit(X)
hidden_states = detector.model.predict(X)


print("Raw hidden states:", {int(state) for state in set(hidden_states)})




Raw hidden states: {0, 1, 2, 3}


In [157]:
# Show regime-labelled hidden states

bull = np.argmax(detector.model.means_[:, 0])
bear = np.argmin(detector.model.means_[:, 0])

remaining = [i for i in range(detector.model.n_components) if i != bull and i != bear]
transition = remaining[np.argmax(detector.model.means_[remaining, 4])]

remaining.remove(transition)
recovery = remaining[0]

regime_labels = {
    bull: 'BULL',
    bear: 'BEAR',
    transition: 'TRANSITION',
    recovery: 'RECOVERY'
}


print("Labelled hidden states:", {int(k): v for k, v in regime_labels.items()})

detector.regime_labels = regime_labels


detector.label_to_state = {v: k for k, v in detector.regime_labels.items()}



Labelled hidden states: {1: 'BULL', 2: 'BEAR', 0: 'TRANSITION', 3: 'RECOVERY'}


#### 4.) The Event-Driven implementation

##### The model is given a **warmup period** enforcing it to accumulate sufficient bars from the backtesting engine before learning begins.

##### A **retrain period** is set to balance computational intensity and data fitting.

##### The **predict()** function runs Viterbi algorithm under the hood, which I have coded from scratch (on my GitHub). It returns a single most probable path, and therefore and integer which represents the most likely state at the time (.iloc[-1])

##### I ended up switching to the **predict_proba()** function which uses the Forward-Backward algorithm to return the **posterior probabilities**.

#### This gave me the model's **confidence** of being in each of the regimes at a given time.


In [163]:
# producing the posterior probabilities

posterior_probs = detector.model.predict_proba(X)
detector.current_probs = posterior_probs[-1]

print(f"BULL: {detector.current_probs[detector.label_to_state['BULL']]:.3f} | BEAR: {detector.current_probs[detector.label_to_state['BEAR']]:.3f} | RECOVERY: {detector.current_probs[detector.label_to_state['RECOVERY']]:.3f} | TRANSITION: {detector.current_probs[detector.label_to_state['TRANSITION']]:.3f}")

state_prob_tuple = detector.get_highest_prob_state()

BULL: 0.097 | BEAR: 0.000 | RECOVERY: 0.077 | TRANSITION: 0.826


##### Using the following weighted formula:

##### Position Size = **1 * P(BULL) + 0 * P(BEAR) + 0.2 * P(TRANSITION) + 0.7 * P(RECOVERY)**

##### I now have a **position size multiplier** based on the model's confidence in being in each regime.

##### This is passed on to my RiskManager class, sizing the position. This dynamic exposure risk management system aims to reduce the flipping that occurs from fixed label-based exposure.

In [159]:
# Position size multiplier

# Here I'm returning it as part of the tuple which is attached to the strategy decision and passed to RiskManager

print(f"Position size: {state_prob_tuple[2]:.3f}") # For viewing purposes this is rounded to 3dp

Position size: 0.324


In [1]:
# Plot regime detector chart
regimes = [detector.regime_labels[state] for state in hidden_states]
fig = go.Figure()

fig.add_trace(go.Scatter(x=spy_df['Date'], y=spy_df['Close'], mode='lines', name='SPY', line=dict(color='white'), line_width=0.5)

regime_colours = {'BULL': 'rgba(34, 197, 94 ,0.15)', 'BEAR': 'rgba(239, 68, 68, 0.15)', 'TRANSITION': 'rgba(234, 179, 8, 0.15)', 'RECOVERY': 'rgba(59, 130, 246, 0.15)'}

start = 0
for i in range(1, len(regimes)):
    if regimes[i] != regimes[start]:
        fig.add_vrect(x0=spy_df['Date'].iloc[start], x1=spy_df['Date'].iloc[i], fillcolor=regime_colours[regimes[start]], opacity=0.5, line_width=0, exclude_empty_subplots=False)
        start = i
        print(i / 5400)

fig.add_vrect(x0=spy_df['Date'].iloc[start], x1=spy_df['Date'].iloc[-1], fillcolor=regime_colours[regimes[start]], opacity=0.5, line_width=0, exclude_empty_subplots=False)

fig.update_layout(title="Market Regime Detection")

for regime, colour in regime_colours.items():
    fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers', marker=dict(size=10, color=colour.replace('0.35', '1')), name=regime))

fig.show()

SyntaxError: '(' was never closed (1552759294.py, line 5)

#### 5.) Limitations To This Example

##### The key **limitation** to this model/example is that the data is prepopulated by me, from inside the notebook.

##### Therefore the model has access to all the data during fitting. This means it's inherently subject to **lookahead bias**.

##### Inside the event-driven backtester I successfully avoid lookahead bias as the model only has access to the bars that precede a given point in time. Results **below**.

#### 6.) Results
Test Date: 2006-01-01 : 2025-01-01  

___________________________________________________________________________________________________________________________________________________________________________________________


##### Buy And Hold Results

Total Portfolio Value: $661758.15

Total return: 561.76%  

Sharpe: 0.61

Max drawdown: 55.33%  

Drawdown duration: 1223.00  

___________________________________________________________________________________________________________________________________________________________________________________________


##### Strategy Results (No regime detection)

Total Portfolio Value: $320358.62  

Total return: 220.36%

Sharpe: 0.65

Max drawdown: 25.39%

Drawdown duration: 684.00

Total Commission: $12027.35

Total Slippage: $30349.62

Wins: 485 | Losses: 432 | Even: 5

Avg Win / Avg Loss: 1.43

Expected Value: 257.79


___________________________________________________________________________________________________________________________________________________________________________________________

#### Strategy Results (With regime detection position sizing)

Total Portfolio Value: $265227.37  

Total return: 165.23%

Sharpe: 0.76

Max drawdown: 12.02%

Drawdown duration: 797.00

Total Commission: $8667.39

Total Slippage: $18928.64

Wins: 452 | Losses: 395 | Even: 5

Avg Win / Avg Loss: 1.58

Expected Value: 208.83

##### Regime-aware position sizing sacrifices raw return for materially better risk-adjusted performance.

##### Max drawdown reduces from 55% (BnH) to 12% while the Sharpe increases to 0.76, the highest of all three tests.

##### This is important as lower drawdowns make the strategy easier to hold through a crash, which is often the reason good retail trading strategies are abandoned.

<br>
<br>


George Sedgwick, 2026